# Summary_Day15_offline.ipynb  
## 사용자 데이터 분류 · Custom Dataset · 인터넷 불가 버전 · ImageFolder 구조 연습

이 파일은 **인터넷이 안 되는 환경**에서 15강의 핵심 구조를 연습하기 위한 버전이다.

원본 강의는 다음을 다운로드한다.

```text
hymenoptera_data.zip
dog_wolf.zip
CIFAR-10
VGG19-BN pretrained weights
ResNet18 ImageNet weights
```

인터넷이 없으면 다운로드가 막힐 수 있다.  
그래서 offline 버전은 직접 작은 이미지 폴더를 생성해서 `ImageFolder` 구조를 연습한다.

```text
랜덤/패턴 이미지 생성
→ train/val 폴더 구조 만들기
→ ImageFolder로 Dataset 생성
→ DataLoader로 batch 공급
→ 작은 CNN으로 학습
→ ResNet18(weights=None)으로 fc 교체 구조 연습
```

> 주의:  
> offline 버전은 진짜 사전학습 성능을 보여주는 파일이 아니다.  
> 인터넷 없이 `ImageFolder`, DataLoader, training loop, fc 교체 구조를 익히는 대체 실습이다.

## 1. 라이브러리 준비

이미지를 직접 만들기 위해 PIL을 사용한다.  
PyTorch와 torchvision은 Dataset, DataLoader, transform, 모델 구조에 사용한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision import models

from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("torchvision:", torchvision.__version__)

## 2. ImageFolder용 폴더 구조 직접 만들기

인터넷 없이도 `ImageFolder`를 연습하려면 직접 폴더를 만들면 된다.

구조는 다음과 같다.

```text
offline_animals/
  train/
    ants/
    bees/
  val/
    ants/
    bees/
```

이미지는 간단한 색상/패턴 이미지로 만든다.

In [ ]:
root_dir = Path("offline_animals")

if root_dir.exists():
    shutil.rmtree(root_dir)

classes = ["ants", "bees"]
splits = ["train", "val"]

for split in splits:
    for class_name in classes:
        (root_dir / split / class_name).mkdir(parents=True, exist_ok=True)

print("폴더 생성 완료:", root_dir)

## 3. 가짜 이미지 저장 함수

ants와 bees를 구분할 수 있도록 서로 다른 패턴을 가진 이미지를 만든다.

In [ ]:
def make_pattern_image(class_name, image_size=224, seed=0):
    rng = np.random.default_rng(seed)

    if class_name == "ants":
        base_color = (120, 70, 40)
        accent_color = (30, 20, 10)
    else:
        base_color = (230, 180, 40)
        accent_color = (30, 30, 30)

    image = Image.new("RGB", (image_size, image_size), base_color)
    draw = ImageDraw.Draw(image)

    for _ in range(12):
        x0 = int(rng.integers(0, image_size - 30))
        y0 = int(rng.integers(0, image_size - 30))
        x1 = x0 + int(rng.integers(10, 50))
        y1 = y0 + int(rng.integers(10, 50))

        if class_name == "ants":
            draw.ellipse([x0, y0, x1, y1], fill=accent_color)
        else:
            draw.rectangle([x0, y0, x1, y1], fill=accent_color)

    noise = rng.normal(0, 10, size=(image_size, image_size, 3))
    arr = np.array(image).astype(np.float32) + noise
    arr = np.clip(arr, 0, 255).astype(np.uint8)

    return Image.fromarray(arr)


for split in splits:
    n_per_class = 30 if split == "train" else 10

    for class_name in classes:
        for i in range(n_per_class):
            img = make_pattern_image(class_name, seed=1000 + i + (0 if class_name == "ants" else 100))
            img.save(root_dir / split / class_name / f"{class_name}_{i:03d}.jpg")

print("이미지 생성 완료")

for path in sorted(root_dir.glob("*/*")):
    print(path, len(list(path.glob("*.jpg"))))

## 4. transforms와 ImageFolder

강의 원본과 같은 흐름으로 학습용 transform과 검증용 transform을 분리한다.

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.5,
        scale=(0.02, 0.33),
        ratio=(0.3, 3.3),
        value=0,
        inplace=False
    ),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.ImageFolder(root_dir / "train", transform=train_transform)
train_data_no_aug = datasets.ImageFolder(root_dir / "train", transform=test_transform)
val_data = datasets.ImageFolder(root_dir / "val", transform=test_transform)

print("classes:", train_data.classes)
print("class_to_idx:", train_data.class_to_idx)
print("train:", len(train_data))
print("val:", len(val_data))

## 5. DataLoader와 이미지 확인

`ImageFolder`가 만든 Dataset을 DataLoader로 묶는다.

In [ ]:
batch_size = 8

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

preview_loader = DataLoader(train_data_no_aug, batch_size=16, shuffle=True)

images, labels = next(iter(preview_loader))

plt.figure(figsize=(10, 4))

for i in range(min(16, len(images))):
    plt.subplot(2, 8, i + 1)

    img = images[i].numpy().transpose((1, 2, 0))
    img = (img + 1) / 2
    img = np.clip(img, 0, 1)

    plt.imshow(img)
    plt.title(train_data.classes[labels[i].item()], fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 6. 오프라인용 작은 CNN 모델

진짜 사전학습 모델 대신 빠르게 실행되는 작은 CNN으로 학습 루프를 확인한다.

In [ ]:
class SmallImageClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SmallImageClassifier(num_classes=2).to(device)

print(model)

## 7. 학습/평가 함수

온라인 버전과 같은 기본 구조다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        pred = torch.max(outputs, 1)[1]

        total_loss += loss.item() * labels.size(0)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            pred = torch.max(outputs, 1)[1]

            total_loss += loss.item() * labels.size(0)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

## 8. 작은 CNN 학습 실행

가짜 데이터라 성능 자체보다 `ImageFolder → DataLoader → 학습` 흐름을 보는 것이 목적이다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = []

for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, y_pred, y_true = evaluate(model, val_loader, criterion, device)

    history.append([epoch + 1, train_loss, train_acc, val_loss, val_acc])

    print(
        f"epoch {epoch + 1} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

history = np.array(history)

In [ ]:
plt.plot(history[:, 0], history[:, 1], label="train loss")
plt.plot(history[:, 0], history[:, 3], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Offline Custom Dataset Loss")
plt.legend()
plt.show()

plt.plot(history[:, 0], history[:, 2], label="train acc")
plt.plot(history[:, 0], history[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Offline Custom Dataset Accuracy")
plt.legend()
plt.show()

## 9. Confusion Matrix와 Classification Report

오프라인에서도 평가 흐름은 동일하다.

In [ ]:
val_loss, val_acc, y_pred, y_true = evaluate(model, val_loader, criterion, device)

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Offline Custom Dataset Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(2), train_data.classes)
plt.yticks(range(2), train_data.classes)

for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar()
plt.show()

print(classification_report(y_true, y_pred, target_names=train_data.classes, zero_division=0))

## 10. ResNet18 fc 교체 구조 연습

인터넷이 없으면 pretrained weight를 다운로드할 수 없다.  
그래도 `weights=None`으로 ResNet18 구조를 만들고 `fc` 교체를 연습할 수 있다.

In [ ]:
resnet = models.resnet18(weights=None)

print("기존 fc:")
print(resnet.fc)

num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 10)

print("\n교체 후 fc:")
print(resnet.fc)

## 11. EarlyStopping 구조 연습

검증 손실이 개선되지 않으면 중단하는 클래스다.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0.0, path="offline_best_model.pth"):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)

        elif score < self.best_score + self.delta:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

early_stopping = EarlyStopping(patience=3)

print("EarlyStopping 구조 준비 완료")

## 12. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `ImageFolder` | 폴더 구조를 Dataset으로 변환 | `datasets.ImageFolder(root)` |
| `class_to_idx` | class 이름과 label 매핑 | 자동 생성 |
| `DataLoader` | batch 공급 | `DataLoader(dataset)` |
| `RandomErasing` | 일부 영역 지우기 | Tensor 뒤에 적용 |
| `SmallImageClassifier` | 오프라인용 작은 CNN | 빠른 구조 연습 |
| `weights=None` | pretrained weight 없이 모델 생성 | 인터넷 불가 구조 연습 |
| `fc` | ResNet 마지막 layer | class 수에 맞게 교체 |
| `EarlyStopping` | 조기 종료 | val loss 개선 없으면 중단 |

## 13. 시험용 요약

```text
오프라인 15강 핵심 = 직접 만든 폴더 이미지로 ImageFolder 구조를 연습한다
```

꼭 기억할 것:

- `ImageFolder`는 폴더 이름을 label로 사용한다.
- 인터넷이 없어도 직접 폴더와 이미지를 만들면 ImageFolder를 연습할 수 있다.
- 학습 데이터에는 랜덤 증강을 넣을 수 있다.
- 검증 데이터에는 랜덤 증강을 넣지 않는다.
- `DataLoader`는 Dataset을 batch로 공급한다.
- `weights=None`은 pretrained weight 없이 모델 구조만 만든다는 뜻이다.
- ResNet18의 마지막 layer는 `fc`다.
- EarlyStopping은 검증 손실이 좋아지지 않으면 학습을 멈춘다.